In [2]:
from pydantic import BaseModel , Field
from langchain.chat_models import ChatOpenAI
from langchain.output_parsers import PydanticOutputParser
from pymongo import MongoClient

In [28]:
from typing import List

class Condition(BaseModel):
    field: str = Field(..., description="The field specified")
    operator: str = Field(..., description="The operator to use [==,!=,>,<,>=,<=]")
    value: str = Field(..., description="The value to compare with")

class Outcome(BaseModel):
    field: str = Field(..., description="The claim field")
    value: str = Field(..., description="The value")

class Rule(BaseModel):
    conditions: List[Condition] = Field(..., description="List of conditions")
    condition_operator: str = Field(..., description="The operator to use [and, or, etc.]")
    outcome: Outcome

class Rules(BaseModel):
    rules: List[Rule] = Field(..., description="List of rules")    
   


In [22]:
llm = ChatOpenAI(temperature=0)
parser = PydanticOutputParser(pydantic_object=Rules)
format_instructions = parser.get_format_instructions()
print(format_instructions)


The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"$defs": {"Condition": {"properties": {"field": {"description": "The field specified", "title": "Field", "type": "string"}, "operator": {"description": "The operator to use [==,!=,>,<,>=,<=]", "title": "Operator", "type": "string"}, "value": {"description": "The value to compare with", "title": "Value", "type": "string"}}, "required": ["field", "operator", "value"], "title": "Condition", "type": "object"}, "Outcome": {"properties": {"field": {"description": "The claim field", "title": "Field", "type": "string"}, "value": {"description": "The 

In [24]:
query = """if deep frying oil is selected and if claim value is less than 100000 percentage is 10 
if deep frying oil is selected and if claim value is more than or equal to 100000 percentage is 5"""

In [23]:
import langchain
langchain.verbose=True
langchain.debug=True

In [25]:

template=f"Answer the user query.\n{format_instructions}\n{query}\n"


In [26]:
final_model = llm | parser

In [20]:
final_model.invoke(template)

[chain/start] [1:chain:RunnableSequence] Entering Chain run with input:
{
  "input": "Answer the user query.\nThe output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {\"properties\": {\"foo\": {\"title\": \"Foo\", \"description\": \"a list of strings\", \"type\": \"array\", \"items\": {\"type\": \"string\"}}}, \"required\": [\"foo\"]}\nthe object {\"foo\": [\"bar\", \"baz\"]} is a well-formatted instance of the schema. The object {\"properties\": {\"foo\": [\"bar\", \"baz\"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{\"$defs\": {\"Condition\": {\"properties\": {\"field\": {\"description\": \"The field specified\", \"title\": \"Field\", \"type\": \"string\"}, \"operator\": {\"description\": \"The operator to use [==,!=,>,<,>=,<=]\", \"title\": \"Operator\", \"type\": \"string\"}, \"value\": {\"description\": \"The value to compare with\", \"title\": \"Value\", \"type\": \"string\"}}, \"required\": [\"f

ValidationError: 2 validation errors for Rules
rules.0.conditions
  Input should be a valid dictionary or instance of Condition [type=model_type, input_value=[{'field': 'selected_opti...'<', 'value': '100000'}], input_type=list]
    For further information visit https://errors.pydantic.dev/2.5/v/model_type
rules.1.conditions
  Input should be a valid dictionary or instance of Condition [type=model_type, input_value=[{'field': 'selected_opti...>=', 'value': '100000'}], input_type=list]
    For further information visit https://errors.pydantic.dev/2.5/v/model_type

In [11]:
from pymongo import MongoClient 
import os
import json 
connection_string = os.getenv("MONGO_CONNECTION_STRING")
# Function to save Pydantic model to MongoDB
def save_to_mongodb(rules: Rules, db_name: str, collection_name: str):
    # Connect to MongoDB (update host and port if needed)
    client = MongoClient(connection_string)
    
    # Select the database and collection
    db = client[db_name]
    collection = db[collection_name]

    # Serialize Pydantic model to JSON, then load it as a dictionary
    rules_dict = json.loads(rules.model_dump_json())

    # Insert the data into MongoDB
    collection.insert_one(rules_dict)

In [27]:
output = final_model.invoke(template)
save_to_mongodb(rules=output, db_name="SWG", collection_name="Rules")

[chain/start] [1:chain:RunnableSequence] Entering Chain run with input:
{
  "input": "Answer the user query.\nThe output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {\"properties\": {\"foo\": {\"title\": \"Foo\", \"description\": \"a list of strings\", \"type\": \"array\", \"items\": {\"type\": \"string\"}}}, \"required\": [\"foo\"]}\nthe object {\"foo\": [\"bar\", \"baz\"]} is a well-formatted instance of the schema. The object {\"properties\": {\"foo\": [\"bar\", \"baz\"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{\"$defs\": {\"Condition\": {\"properties\": {\"field\": {\"description\": \"The field specified\", \"title\": \"Field\", \"type\": \"string\"}, \"operator\": {\"description\": \"The operator to use [==,!=,>,<,>=,<=]\", \"title\": \"Operator\", \"type\": \"string\"}, \"value\": {\"description\": \"The value to compare with\", \"title\": \"Value\", \"type\": \"string\"}}, \"required\": [\"f